Imports and Paths

In [3]:
import pandas as pd
import warnings

# -----------------
# !! ACTION REQUIRED: Update this path to your dataset file !!
# -----------------
# PASTE YOUR FULL FILE PATH HERE. 
# Use 'r' before the string to handle backslashes (Windows)
# 
# Windows Example: r'C:\Users\YourName\Documents\nhanes_merged.csv'
# Mac/Linux Example: '/Users/YourName/Documents/nhanes_merged.csv'
# Colab Example: '/content/nhanes_merged.csv'
#
DATASET_PATH = r'C:\diabetes_prediction_project\data\02_intermediate\nhanes_merged.csv' 
# -----------------

# Suppress warnings if any (e.g., from .XPT files)
warnings.filterwarnings('ignore')

try:
    if DATASET_PATH.lower().endswith('.csv'):
        df = pd.read_csv(DATASET_PATH)
    elif DATASET_PATH.lower().endswith('.xpt'):
        print("Reading SAS .XPT file...")
        df = pd.read_sas(DATASET_PATH)
    else:
        # Try CSV by default if extension is unknown
        print("Unknown extension, trying to read as CSV...")
        df = pd.read_csv(DATASET_PATH)
        
except FileNotFoundError:
    print(f"--- ERROR: FILE NOT FOUND ---")
    print(f"Could not find the file at the path you specified: {DATASET_PATH}")
    print("Please double-check the full file path and try again.")
    df = None
except Exception as e:
    print(f"--- ERROR: FAILED TO LOAD FILE ---")
    print(f"Found the file, but failed to read it. Error: {e}")
    print("If it's not a CSV, ensure it's a SAS .XPT file and the name ends in .xpt")
    df = None

if df is not None:
    print("\n----------- SUCCESS: DataFrame .info() -----------")
    df.info()
    
    print("\n\n--------- DataFrame .head() [Sample Data] ---------")
    print(df.head())
    
    print("\n\n--------- !! KEY: Column Headers !! ---------")
    print("Please paste this list in your reply:")
    print(df.columns.tolist())


----------- SUCCESS: DataFrame .info() -----------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25531 entries, 0 to 25530
Columns: 548 entries, SEQN to LBDLDNSI
dtypes: float64(545), object(3)
memory usage: 106.7+ MB


--------- DataFrame .head() [Sample Data] ---------
      SEQN  SDDSRVYR  RIDSTATR  RIAGENDR  RIDAGEYR  RIDAGEMN  RIDRETH1  \
0  83732.0       9.0       2.0       1.0      62.0       NaN       3.0   
1  83733.0       9.0       2.0       1.0      53.0       NaN       3.0   
2  83734.0       9.0       2.0       1.0      78.0       NaN       3.0   
3  83735.0       9.0       2.0       2.0      56.0       NaN       3.0   
4  83736.0       9.0       2.0       2.0      42.0       NaN       4.0   

   RIDRETH3  RIDEXMON  RIDEXAGM  ...  MCQ366C  MCQ366D  MCQ371A  MCQ371B  \
0       3.0       1.0       NaN  ...      NaN      NaN      NaN      NaN   
1       3.0       1.0       NaN  ...      NaN      NaN      NaN      NaN   
2       3.0       2.0       NaN  ...      NaN     

In [ ]:
import pandas as pd
import os
import pyreadstat
import numpy as np

# --- Define the paths to your raw data folders based on your tree structure ---
NHANES_2015_2016_DIR = r'../data/01_raw/NHANES/2015-2016'
NHANES_2017_2020_DIR = r'../data/01_raw/NHANES/2017-March 2020 Pre-Pandemic'

print("Paths defined successfully.")
print("--- CHECKPOINT 1/5: Imports and paths are set. ---")

ModuleNotFoundError: No module named 'pyreadstat.pyreadstat'

Merging Function

In [2]:
def load_and_merge_nhanes_cycle(folder_path, demo_filename):
    """
    Loads all .xpt files from a given NHANES cycle folder and merges them
    into a single pandas DataFrame on the 'SEQN' column.
    """
    print(f"\nProcessing files in: {folder_path}")
    xpt_files = [f for f in os.listdir(folder_path) if f.lower().endswith('.xpt')]
    base_df, meta = pyreadstat.read_xport(os.path.join(folder_path, demo_filename), encoding='latin1')
    print(f"  - Loaded base file: {demo_filename} with {len(base_df)} rows.")
    
    for filename in xpt_files:
        if filename.lower() != demo_filename.lower():
            file_path = os.path.join(folder_path, filename)
            temp_df, meta = pyreadstat.read_xport(file_path, encoding='latin1')
            cols_to_drop = [col for col in temp_df.columns if col in base_df.columns and col != 'SEQN']
            temp_df = temp_df.drop(columns=cols_to_drop)
            base_df = pd.merge(base_df, temp_df, on='SEQN', how='left')
            print(f"  - Merged file: {filename}")
            
    return base_df

print("--- CHECKPOINT 2/5: Merging function is defined. ---")

--- CHECKPOINT 2/5: Merging function is defined. ---


Execute NHANES Merge

In [3]:
df_2015_2016 = load_and_merge_nhanes_cycle(NHANES_2015_2016_DIR, 'DEMO_I.xpt')
df_2017_2020 = load_and_merge_nhanes_cycle(NHANES_2017_2020_DIR, 'P_DEMO.xpt')

nhanes_full_df = pd.concat([df_2015_2016, df_2017_2020], ignore_index=True)

output_path = '../data/02_intermediate/nhanes_merged.csv'
nhanes_full_df.to_csv(output_path, index=False)

print(f"\nSuccessfully saved the merged NHANES data to: {output_path}")
print("--- CHECKPOINT 3/5: NHANES files merged and saved to intermediate folder. ---")


Processing files in: ../data/01_raw/NHANES/2015-2016
  - Loaded base file: DEMO_I.xpt with 9971 rows.
  - Merged file: BIOPRO_I.xpt
  - Merged file: BMX_I.xpt
  - Merged file: BPQ_I.xpt
  - Merged file: BPX_I.xpt
  - Merged file: DIQ_I.xpt
  - Merged file: FASTQX_I.xpt
  - Merged file: GHB_I.xpt
  - Merged file: GLU_I.xpt
  - Merged file: HDL_I.xpt
  - Merged file: INS_I.xpt
  - Merged file: MCQ_I.xpt
  - Merged file: OGTT_I.xpt
  - Merged file: PAQ_I.xpt
  - Merged file: SMQ_I.xpt
  - Merged file: TCHOL_I.xpt
  - Merged file: TRIGLY_I.xpt
  - Merged file: WHQ_I.xpt

Processing files in: ../data/01_raw/NHANES/2017-March 2020 Pre-Pandemic
  - Loaded base file: P_DEMO.xpt with 15560 rows.
  - Merged file: P_BIOPRO.xpt
  - Merged file: P_BMX.xpt
  - Merged file: P_BPQ.xpt
  - Merged file: P_BPXO.xpt
  - Merged file: P_DIQ.xpt
  - Merged file: P_FASTQX.xpt
  - Merged file: P_GHB.xpt
  - Merged file: P_GLU.xpt
  - Merged file: P_HDL.xpt
  - Merged file: P_INS.xpt
  - Merged file: P_MCQ.xpt

Clean and Save Final NHANES Data 

In [4]:
import numpy as np

# --- 1. Select all columns needed ---
columns_to_keep = [
    'SEQN', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'BMXBMI', 'BPXSY1', 'BPXDI1', 
    'SMQ020', 'PAQ650', 'MCQ160C', 'MCQ160F', 'DIQ010', 'DIQ050', 
    'LBXTC', 'LBDHDD', 'LBXGH', 'LBXGLU'
]
columns_to_keep = [col for col in columns_to_keep if col in nhanes_full_df.columns]
nhanes_selected_df = nhanes_full_df[columns_to_keep].copy()


# --- 2. *** APPLY AGE FILTER HERE (THE CORRECTION) *** ---
# This is the new, correct position for the age filter.
nhanes_selected_df = nhanes_selected_df[nhanes_selected_df['RIDAGEYR'] >= 18].copy()


# --- 3. Create the 'Diabetes_Outcome' variable ---
nhanes_selected_df['Diabetes_Outcome'] = np.where(
    (nhanes_selected_df['DIQ010'] == 1) | 
    (nhanes_selected_df['LBXGH'] >= 6.5) | 
    (nhanes_selected_df['LBXGLU'] >= 126), 
    1, 0
)

# --- 4. Create the final feature set by dropping leaky/ID columns ---
final_feature_columns = [
    'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'BMXBMI', 'BPXSY1', 'BPXDI1', 
    'SMQ020', 'PAQ650', 'MCQ160C', 'MCQ160F', 'DIQ050', 'LBXTC', 'LBDHDD'
]
nhanes_final_df = nhanes_selected_df[final_feature_columns + ['Diabetes_Outcome']].copy()

# --- 5. Rename the final columns ---
rename_dict = {
    'RIDAGEYR': 'Age', 'RIAGENDR': 'Gender', 'RIDRETH3': 'Race_Ethnicity',
    'BMXBMI': 'BMI', 'BPXSY1': 'Systolic_BP', 'BPXDI1': 'Diastolic_BP',
    'SMQ020': 'Smoking_Status', 'PAQ650': 'Physical_Activity',
    'MCQ160C': 'History_Heart_Attack', 'MCQ160F': 'History_Stroke',
    'DIQ050': 'Family_History_Diabetes', 'LBXTC': 'Total_Cholesterol',
    'LBDHDD': 'HDL_Cholesterol'
}
nhanes_final_df = nhanes_final_df.rename(columns=rename_dict)

# --- 6. Save the final, clean file ---
output_path_final = '../data/03_processed/nhanes_final.csv'
nhanes_final_df.to_csv(output_path_final, index=False)

print(f"Final, non-leaky NHANES feature set created and saved to {output_path_final}")
print("Verification: Minimum age in final dataset is:", nhanes_final_df['Age'].min())

Final, non-leaky NHANES feature set created and saved to ../data/03_processed/nhanes_final.csv
Verification: Minimum age in final dataset is: 18.0
